# PT Progress Tracker - Pose Model Training (Subset)

This notebook trains a **YOLOv8-pose** model for the PT Progress Tracker application.

## What This Does
1. Downloads the COCO keypoints dataset (Ultralytics auto-downloads it)
2. Fine-tunes a YOLOv8-pose model on a **10% subset** of COCO (~12K images)
3. Validates the trained model on COCO val2017
4. Exports the weights for use in the PT Tracker app

## Why a Subset?
Training on the full COCO dataset (118K images) for 100 epochs takes 30+ hours on a
free Colab T4 GPU. By using a 10% subset and fewer epochs, we can train in ~40-50
minutes while still producing a usable model. Since we start from pretrained
YOLOv8n-pose weights (already trained on COCO), even a short fine-tuning run
produces a model that performs decently for pose estimation.

## Instructions
- Make sure your Colab runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU)
- Run each cell in order
- Training takes ~40-50 minutes on a free T4 GPU
- Download the final `pttracker-pose.pt` file when done

In [ ]:
#@title 1. Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

In [ ]:
#@title 2. Install Ultralytics
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

## 3. Training Configuration

We use a **10% fraction** of the COCO training set. This means only ~12K images
are used per epoch instead of the full ~118K, making training ~10x faster.

Key parameters:
- **fraction=0.1**: Use 10% of training images each epoch
- **epochs=30**: Enough for fine-tuning from pretrained weights
- **patience=10**: Early stopping if no improvement for 10 epochs
- **close_mosaic=15**: Disable mosaic augmentation for last 15 epochs for cleaner final weights
- **lr0=0.005**: Lower learning rate suitable for fine-tuning
- **cache=False**: Read images from disk each epoch (avoids OOM on Colab free tier — the full COCO dataset is too large to fit in ~12GB RAM even with fraction=0.1)

In [ ]:
#@title Training Parameters

MODEL_SIZE = 'n'  #@param ['n']
EPOCHS = 30  #@param {type:"slider", min:10, max:100, step:5}
IMGSZ = 640  #@param [416, 640]
BATCH = -1  #@param {type:"integer"}
LR0 = 0.005  #@param {type:"number"}
FRACTION = 0.1  #@param {type:"slider", min:0.05, max:0.5, step:0.05}
CLOSE_MOSAIC = 15  #@param {type:"integer"}
PATIENCE = 10  #@param {type:"integer"}

PRETRAINED_MODEL = f'yolov8{MODEL_SIZE}-pose.pt'
OUTPUT_NAME = f'pttracker-pose-{MODEL_SIZE}'

print(f'Base model: {PRETRAINED_MODEL}')
print(f'Output name: {OUTPUT_NAME}')
print(f'Epochs: {EPOCHS}')
print(f'Fraction of COCO: {FRACTION} (~{int(118287 * FRACTION)} images/epoch)')
print(f'Image size: {IMGSZ}')
print(f'Learning rate: {LR0}')
print(f'Close mosaic: last {CLOSE_MOSAIC} epochs')
print(f'Early stopping patience: {PATIENCE}')
print(f'\nEstimated training time: ~{EPOCHS} min on T4 (plus ~10 min COCO download on first run)')

In [ ]:
#@title 4. Run Training

model = YOLO(PRETRAINED_MODEL)

results = model.train(
    data='coco-pose.yaml',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    lr0=LR0,
    fraction=FRACTION,
    close_mosaic=CLOSE_MOSAIC,
    project='pttracker',
    name=OUTPUT_NAME,
    patience=PATIENCE,
    cache=False,
    verbose=True,
)

print('\nTraining complete!')

In [ ]:
#@title 5. Validate Model

best_model_path = f'pttracker/{OUTPUT_NAME}/weights/best.pt'
best_model = YOLO(best_model_path)

metrics = best_model.val(data='coco-pose.yaml', imgsz=IMGSZ)

print(f'\nValidation Results:')
print(f'  Box mAP50:    {metrics.box.map50:.4f}')
print(f'  Box mAP50-95: {metrics.box.map:.4f}')
print(f'  Pose mAP50:    {metrics.pose.map50:.4f}')
print(f'  Pose mAP50-95: {metrics.pose.map:.4f}')
print(f'\nNote: These metrics are on the full COCO val set.')
print(f'Performance is expected to be lower than the pretrained model,')
print(f'but should be sufficient for the PT Tracker use case.')

In [ ]:
#@title 6. Test on a Sample Image

import glob

sample_images = glob.glob('pttracker/*/val_batch*.jpg')
if sample_images:
    from IPython.display import Image, display
    latest = sorted(sample_images)[-1]
    print(f'Displaying: {latest}')
    display(Image(filename=latest, width=800))
else:
    print('No validation images found. Running a quick prediction...')
    test_result = best_model.predict(source='https://ultralytics.com/images/bus.jpg', save=True)
    saved_path = test_result[0].save_dir
    import os
    for f in os.listdir(saved_path):
        if f.endswith('.jpg'):
            display(Image(filename=os.path.join(saved_path, f), width=800))
            break

In [ ]:
#@title 7. Export Final Weights

import shutil
import os

best_weights = f'pttracker/{OUTPUT_NAME}/weights/best.pt'
export_name = 'pttracker-pose.pt'

shutil.copy2(best_weights, export_name)
print(f'Exported: {export_name}')
print(f'File size: {os.path.getsize(export_name) / 1e6:.1f} MB')

try:
    from google.colab import files
    files.download(export_name)
    print('\nDownload started! Place the file at ~/.pttracker/pttracker-pose.pt on your machine.')
except ImportError:
    print('\nNot running in Colab. The file is at: ' + os.path.abspath(export_name))
    print('Copy it to ~/.pttracker/pttracker-pose.pt on your local machine.')

## Setup After Download

Once you've downloaded `pttracker-pose.pt`, place it on your local machine:

```bash
mkdir -p ~/.pttracker
mv ~/Downloads/pttracker-pose.pt ~/.pttracker/pttracker-pose.pt
```

The PT Tracker app will automatically find and use this model file. If the file
is missing, the app falls back to the standard `yolov8n-pose.pt` pretrained model
(auto-downloaded by Ultralytics).